In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ssl

# Fix SSL certificate issue on Mac
ssl._create_default_https_context = ssl._create_unverified_context

tickers = ["SPY", "GLD", "XOM", "TLT", "VNQ"]
# SPY = S&P500 | GLD = Gold | XOM = Energy | TLT = Long Bonds | VNQ = Real Estate

raw = yf.download(tickers, start="2019-01-01", end="2024-12-31", auto_adjust=True)
prices = raw["Close"]
prices.to_csv("stock_prices.csv")

print("=== STOCK PRICES ===")
print(prices.shape)
print(prices.head())

cpi = pd.read_csv(
    "cpi_data.csv", parse_dates=["observation_date"], index_col="observation_date"
)
cpi.columns = ["CPI"]
cpi = cpi["2019-01-01":"2024-12-31"]
cpi.to_csv("cpi_data.csv")

print("\n=== CPI DATA ===")
print(cpi.shape)
print(cpi.tail())

print("\n=== STOCK PRICES INSPECTION ===")
print(prices.info())
print(prices.isnull().sum())
print(prices.describe().round(2))

print("\n=== CPI INSPECTION ===")
print(cpi.describe().round(2))

[*********************100%***********************]  5 of 5 completed

=== STOCK PRICES ===
(1509, 5)
Ticker             GLD         SPY         TLT        VNQ        XOM
Date                                                                
2019-01-02  121.330002  224.382523   99.080139  55.523548  50.001842
2019-01-03  122.430000  219.028122  100.207626  55.911709  49.234135
2019-01-04  121.440002  226.364639   99.047714  56.566273  51.049366
2019-01-07  121.860001  228.149475   98.755684  57.121899  51.314850
2019-01-08  121.529999  230.292984   98.496147  58.301609  51.687943

=== CPI DATA ===
(72, 1)
                      CPI
observation_date         
2024-08-01        314.062
2024-09-01        314.732
2024-10-01        315.631
2024-11-01        316.528
2024-12-01        317.604

=== STOCK PRICES INSPECTION ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1509 entries, 2019-01-02 to 2024-12-30
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GLD     1509 non-null   float64
 1   SPY  

In [4]:
# Reindex to full calendar days and forward fill
full_idx = pd.date_range(prices.index.min(), prices.index.max(), freq="D")
prices = prices.reindex(full_idx).ffill()

print(f"Rows after reindex: {len(prices)}")
print(f"Nulls remaining: {prices.isnull().sum().sum()}")

Rows after reindex: 2190
Nulls remaining: 0


In [5]:
# Resample monthly CPI to daily frequency
cpi_daily = cpi.reindex(prices.index, method="ffill")

print(f"CPI daily shape: {cpi_daily.shape}")
print(cpi_daily.head(10))

CPI daily shape: (2190, 1)
                CPI
2019-01-02  252.561
2019-01-03  252.561
2019-01-04  252.561
2019-01-05  252.561
2019-01-06  252.561
2019-01-07  252.561
2019-01-08  252.561
2019-01-09  252.561
2019-01-10  252.561
2019-01-11  252.561


In [6]:
# Simple daily returns
simple_returns = prices.pct_change().dropna()

# Log returns
log_returns = np.log(prices / prices.shift(1)).dropna()

# Monthly returns
monthly_prices = prices.resample("MS").first()
monthly_returns = monthly_prices.pct_change().dropna()

print("Simple returns shape:", simple_returns.shape)
print(simple_returns.head(3).round(4))

Simple returns shape: (2189, 5)
Ticker         GLD     SPY     TLT     VNQ     XOM
2019-01-03  0.0091 -0.0239  0.0114  0.0070 -0.0154
2019-01-04 -0.0081  0.0335 -0.0116  0.0117  0.0369
2019-01-05  0.0000  0.0000  0.0000  0.0000  0.0000


In [7]:
# Monthly CPI change (inflation rate)
cpi_monthly = cpi.resample("MS").first()
inflation_rate = cpi_monthly.pct_change().dropna()
inflation_rate.columns = ["inflation"]

# Align dates
combined = monthly_returns.join(inflation_rate, how="inner")

# Real return formula: (1+nominal) / (1+inflation) - 1
for col in ["SPY", "GLD", "XOM", "TLT", "VNQ"]:
    combined[f"{col}_real"] = (1 + combined[col]) / (1 + combined["inflation"]) - 1

print(combined[["SPY", "SPY_real", "inflation"]].head())

                 SPY  SPY_real  inflation
2019-02-01  0.079463  0.076233   0.003001
2019-03-01  0.038362  0.034450   0.003782
2019-04-01  0.023782  0.019947   0.003760
2019-05-01  0.020922  0.020670   0.000247
2019-06-01 -0.056681 -0.056374  -0.000325
